### Different Models access using Langchain

In [2]:
import os
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')
from langchain.chat_models import init_chat_model
model = init_chat_model('groq:llama-3.3-70b-versatile')
response = model.invoke('Hello what is the whether in hyderabad')
response.content


"Hello! Hyderabad, being a city located in the southern part of India, has a tropical wet and dry climate. The weather in Hyderabad can vary depending on the time of year.\n\n**Current Weather:**\nI'm a large language model, I don't have real-time access to current weather conditions. However, I can suggest some ways for you to find out the current weather in Hyderabad:\n\n1. Check online weather websites: You can check websites like AccuWeather, Weather.com, or the Indian Meteorological Department (IMD) website for the current weather conditions in Hyderabad.\n2. Mobile Apps: You can download mobile apps like Dark Sky, Weather Underground, or Mausam (by IMD) to get the current weather conditions in Hyderabad.\n3. Local News: You can also check local news websites or TV channels for the current weather conditions in Hyderabad.\n\n**General Climate:**\nHyderabad has a hot and dry climate during the summer months (March to May), with temperatures often reaching 40°C (104°F) or more. The 

In [3]:
response = model.invoke('Yes i would like to know more about the whether in Hyd..')
response.content

"You're interested in knowing more about the weather in Hyderabad. Hyderabad, being the capital city of Telangana, India, has a tropical wet and dry climate. Here's an overview of the typical weather conditions in Hyderabad:\n\n**Seasons:**\n\n1. **Summer (March to May):** This is the hottest season in Hyderabad, with temperatures often reaching 40°C (104°F) or more. The heat is intense, and the humidity is relatively low.\n2. **Monsoon (June to September):** The monsoon season brings significant rainfall to Hyderabad, with most of the city's annual rainfall occurring during these months. The temperatures are relatively cooler, ranging from 25°C to 30°C (77°F to 86°F).\n3. **Winter (December to February):** Winters in Hyderabad are mild, with temperatures ranging from 15°C to 25°C (59°F to 77°F). The humidity is relatively low, making it a pleasant time to visit the city.\n4. **Post-monsoon (October to November):** This period is characterized by a gradual decrease in temperature and h

In [4]:
os.environ['GOOGLE_API_KEY'] = os.getenv('GOOGLE_API_KEY')
gemini_model = init_chat_model('gemini-3.5-flash',model_provider="google_genai")
response_gemini = gemini_model.invoke('hey how are you my name is anil')
response_gemini.text

"Hi Anil! It's great to meet you. \n\nI'm doing well, thank you for asking! How are you doing today? How can I help you?"

In [5]:
import langchain
langchain.__version__

'1.3.13'

### otherways of integrating model without chat model

In [6]:
###accessing openai using package without chat_model
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model='gpt-5.6')
resp = model.invoke('how are you openAI')
resp.content

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
###accessing grok using package without chat_model
from langchain_groq import ChatGroq
model = ChatGroq(model='llama-3.3-70b-versatile')
resp = model.invoke('how are you Groq')
resp.content

"I'm just a language model, I don't have feelings or emotions like humans do, but I'm functioning properly and ready to assist you. However, I'm curious - who or what is Groq? Is that a character or a reference I should be familiar with?"

In [ ]:
###accessing gemini using package without chat_model
from langchain_google_genai import ChatGoogleGenerativeAI
model = ChatGoogleGenerativeAI(model = 'gemini-3.5-flash')
resp = model.invoke('How are you gemini')
resp.content[0]['text']

"Hello! I'm doing well, thank you for asking. How are you doing today? How can I help you?"

## Batch and streaming

In [ ]:
for chunk in model.stream('How are you gemini') :
    print(chunk.text, end = '|', flush=True)


Hello! I'|m doing really well, thank you for asking. How are you doing today? 

How can I help you?|||

In [ ]:
for chunk in model.batch(['hello what is my name?','how are you doing','do you have emotions']):
    print(chunk.content)    

In [ ]:
##making different kind of tools
import os
from langchain.chat_models import init_chat_model
from langchain.tools import tool
os.environ['GROQ_API_KEY'] = os.getenv('GROQ_API_KEY')

@tool
def check_whether (city:str)->str :
    '''give the whether of a city'''
    return f'whether in {city} is cloudy and sunny...'

new_gemini_model = init_chat_model(model='groq:llama-3.3-70b-versatile')
model_with_tool = new_gemini_model.bind_tools([check_whether])
resp = model_with_tool.invoke('what is the whether like in Hyderabad?')
# print(resp)
for chunk in resp.tool_calls:
    print(f'Tool : {chunk['name']}' )
    print(f'Args: {chunk['args']}')
    print(f'ID: {chunk['id']}')

Tool : check_whether
Args: {'city': 'Hyderabad'}
ID: x8dkj158t


In [ ]:
##Tools execution Loops
message = [{"role":"user","content":"what is the whether in Boston?"}]
ai_msg = model_with_tool.invoke(message)
message.append(ai_msg)

for tool_call in ai_msg.tool_calls:
   tool_resp = check_whether.invoke(tool_call)
   message.append(tool_resp)

final_resp = model_with_tool.invoke(message) 
final_resp.content

'I only provide the function call to get the weather. The actual weather in Boston is provided by the function call result, which is cloudy and sunny.'

In [17]:
import os
from langchain.messages import SystemMessage,HumanMessage,AIMessage
from langchain_groq import ChatGroq
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
s_message = SystemMessage('You are Programmer with 30 Years of Experience in AI')
ai_message = AIMessage('How can i help you ?')
h_message = HumanMessage('explain about rag in 10 lines')
model = ChatGroq(model = 'llama-3.3-70b-versatile')

res = model.invoke([s_message,ai_message,h_message])
res.content

'RAG (Retrieve, Augment, Generate) is a framework for building conversational AI models.\nIt combines retrieval and generation techniques to provide more accurate responses.\nRAG uses a retriever to fetch relevant information from a database or knowledge graph.\nThe augmenter then expands on this information to provide more context.\nThe generator uses this context to produce a final response.\nRAG is particularly useful for tasks that require a deep understanding of language and context.\nIt has been used in applications such as chatbots and virtual assistants.\nRAG can be fine-tuned for specific domains or tasks.\nThis framework has shown promising results in natural language processing tasks.\nOverall, RAG is a powerful tool for building more sophisticated conversational AI systems.'

In [22]:
from langchain.messages import SystemMessage,AIMessage,HumanMessage,ToolMessage
h_msg = HumanMessage('what is the whether in fransisco')

ai_msg = AIMessage(content=[],tool_calls=[{"name":"check_whether","args":{"location":"fransisco"},"id":"anil_123" }])

tool_msg = ToolMessage(content='70 degress and sunny..',tool_call_id = 'anil_123')

messages = [h_msg,ai_msg,tool_msg]
resp = model.invoke(messages)
resp.content

"The current weather in San Francisco is 70 degrees and sunny. San Francisco is known for its mild climate, with temperatures ranging from the mid-50s to the mid-70s (13°C to 24°C) throughout the year. However, the weather can vary depending on the time of year and other factors.\n\nIf you're planning a trip to San Francisco, here's a rough idea of what you can expect:\n\n* Summer (June to August): Warm and sunny, with temperatures in the mid-70s to mid-80s (24°C to 30°C)\n* Fall (September to November): Mild and pleasant, with temperatures in the mid-60s to mid-70s (18°C to 24°C)\n* Winter (December to February): Cool and rainy, with temperatures in the mid-50s to mid-60s (13°C to 18°C)\n* Spring (March to May): Mild and sunny, with temperatures in the mid-60s to mid-70s (18°C to 24°C)\n\nKeep in mind that these are general temperature ranges, and the weather can vary from year to year. It's always a good idea to check the forecast before your trip to get a more accurate idea of the w

In [25]:
## model generating the structured out with pydantic
from pydantic import BaseModel,Field

##class defines the structure of output
class Movie (BaseModel):
    title:str = Field('title of the movie')
    year:int = Field('year of release of movie')
    ratings:float = Field('ratings of the movie')
    director:str = Field('director of the movie')

model_with_structured_output = model.with_structured_output(Movie)##mapped structure class with model
response = model_with_structured_output.invoke('Bahubali')
response

Movie(title='Baahubali', year=2015, ratings=8.1, director='S. S. Rajamouli')

In [42]:
## model generating the structured out with pydantic
from pydantic import BaseModel,Field

##structed as well as row included output
class Movie (BaseModel):
    title:str = Field(..., description='title of the movie')
    year:int = Field(...,description='year of release of movie')
    ratings:float = Field(...,description='ratings of the movie')
    director:str = Field(...,description='director of the movie')

model_with_structured_output = model.with_structured_output(Movie,include_raw=True)##mapped structure class with model
response = model_with_structured_output.invoke('Bahubali')
response["parsed"]

Movie(title='Bahubali', year=2015, ratings=8.1, director='S.S. Rajamouli')

In [45]:
## model generating the structured out with pydantic
from pydantic import BaseModel,Field

##Nested structured output
class Actor (BaseModel):
    name:str
    role:str
    age:int
    birth_place:str

class MovieDetails (BaseModel):
    title:str = Field('Movie Name')
    year:str = Field('Year of realease')
    cast:list[Actor]
    genre:list[str]
    budget:float | None = Field(None, description='budget of the Movie')


model_with_structured_output = model.with_structured_output(MovieDetails)##mapped structure class with model
response = model_with_structured_output.invoke('i want to know the details of movie bahubali')
response

MovieDetails(title='Bahubali', year='2015', cast=[Actor(name='Prabhas', role='Amarendra Baahubali', age=41, birth_place='Madras, Tamil Nadu, India'), Actor(name='Rana Daggubati', role='Bhallaladeva', age=37, birth_place='Madras, Tamil Nadu, India')], genre=['Action', 'Adventure', 'Drama'], budget=250.0)